Question 1: 
The objective of this question is to analyze the fairness and expected earnings in American roulette when consistently betting on black. We apply both theoretical probability and Monte-Carlo simulation to:
<li>Compute the probability of winning/losing per spin</li>
<li>Derive expectation and variance of total earnings</li>
<li>Visualize the distribution of outcomes for different spin counts</li>
<li>Demonstrate convergence behavior predicted by the Law of Large Numbers and Central Limit Theorem</li>


Question 2:
<li>Used the Central Limit Theorem to build 95% confidence intervals for proportions</li>
<li>Simulated polling to check how often CIs capture the true proportion</li>
<li>Analyzed real 2016 U.S. election polling data (Clinton vs. Trump)</li>
<li>Calculated weighted candidate support and confidence intervals</li>
<li>Performed a hypothesis test to determine whether the race was statistically tied</li>

Question 3:
<li>Performed data cleaning and preprocessing on clinical safety data</li>
<li>Conducted descriptive statistics and group stratification</li>
<li>Applied inferential testing to evaluate treatment effects</li>
<li>Found no statistically significant differences between treatment and control groups</li>


In [18]:
import pandas as pd


In [19]:
FILE_NAME = "drug_safety.csv"

In [20]:
dataset = pd.read_csv(FILE_NAME)

In [21]:
dataset.head()

,age,sex,trx,week,wbc,rbc,adverse_effects,num_effects
0,62,male,Drug,0,7.3,5.1,No,0
1,62,male,Drug,1,NaN,NaN,No,0
2,62,male,Drug,12,5.6,5.0,No,0
3,62,male,Drug,16,NaN,NaN,No,0
4,62,male,Drug,2,6.6,5.1,No,0


In [22]:
def missing_values(dataset):
    values_with_nan = [col for col in dataset.columns if dataset[col].isnull().sum() > 0 ]
    return values_with_nan

print(missing_values(dataset))

['wbc', 'rbc']


In [23]:
df = dataset.dropna()

In [24]:
type(df['rbc'])

pandas.core.series.Series

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9127 entries, 0 to 16102
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              9127 non-null   int64  
 1   sex              9127 non-null   object 
 2   trx              9127 non-null   object 
 3   week             9127 non-null   int64  
 4   wbc              9127 non-null   float64
 5   rbc              9127 non-null   float64
 6   adverse_effects  9127 non-null   object 
 7   num_effects      9127 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 641.7+ KB


In [26]:
df

,age,sex,trx,week,wbc,rbc,adverse_effects,num_effects
0,62,male,Drug,0,7.3,5.1,No,0
2,62,male,Drug,12,5.6,5.0,No,0
4,62,male,Drug,2,6.6,5.1,No,0
6,62,male,Drug,4,6.9,5.2,Yes,1
7,62,male,Drug,8,7.1,5.0,Yes,1
...,...,...,...,...,...,...,...,...
16095,78,male,Placebo,0,7.2,5.0,No,0
16097,78,male,Placebo,12,6.5,4.9,No,0
16099,78,male,Placebo,2,7.5,4.9,No,0
16101,78,male,Placebo,4,6.4,4.8,No,0


In [27]:
def display_basic_statics(df):
    mean_values = df.mean(numeric_only = True)
    std_values = df.std(numeric_only = True)
    return mean_values, std_values

mean_values, std_values = display_basic_statics(df)
print("mean values: ", mean_values)
print("std values: ", std_values)


mean values:  age            64.007670
week            4.985428
wbc             7.340331
rbc             4.672784
num_effects     0.100800
dtype: float64
std values:  age            8.847711
week           4.375397
wbc            1.996645
rbc            0.458520
num_effects    0.322178
dtype: float64


In [28]:
def group_by_trx(df):
    df_drug = df[df['trx'] == 'Drug']
    df_placebo = df[df['trx'] == 'Placebo']
    return df_drug, df_placebo
df_drug, df_placebo = group_by_trx(df)
df_drug.info()
df_placebo.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6011 entries, 0 to 16092
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              6011 non-null   int64  
 1   sex              6011 non-null   object 
 2   trx              6011 non-null   object 
 3   week             6011 non-null   int64  
 4   wbc              6011 non-null   float64
 5   rbc              6011 non-null   float64
 6   adverse_effects  6011 non-null   object 
 7   num_effects      6011 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 422.6+ KB
<class 'pandas.core.frame.DataFrame'>
Int64Index: 3116 entries, 32 to 16102
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              3116 non-null   int64  
 1   sex              3116 non-null   object 
 2   trx              3116 non-null   object 
 3   week             3116 non-

In [35]:
def key_statistics(df):  #check if other statics are needed
    mean_val_wbc = df['wbc'].mean(numeric_only = True)
    std_val_wbc = df['wbc'].std(numeric_only = True)
    
    mean_val_rbc = df['rbc'].mean(numeric_only = True)
    std_val_rbc = df['rbc'].std(numeric_only = True)
    
    mean_val_effects = df['num_effects'].mean(numeric_only = True)
    std_val_effects = df['num_effects'].std(numeric_only = True)
    
    stats_df = pd.DataFrame({
        'wbc': [mean_val_wbc, std_val_wbc],
        'rbc': [mean_val_rbc, std_val_rbc],
        'num_effects': [mean_val_effects, std_val_effects]
    }, index=['mean', 'std'])  
    
    return stats_df
stats_df_drug = key_statistics(df_drug)

stats_df_placebo = key_statistics(df_placebo)
print("Stats of Drug Group: ", stats_df_drug)
print("Stats of Placebo Group: ", stats_df_placebo)


Stats of Drug Group:             wbc      rbc  num_effects
mean  7.330461  4.67913     0.102479
std   2.008941  0.45499     0.325529
Stats of Placebo Group:             wbc       rbc  num_effects
mean  7.359371  4.660542     0.097561
std   1.972890  0.465083     0.315639


In [30]:
def numerical_adverse_effects(df):
    df = df.copy()  
    df.loc[:, 'adverse_effects'] = df['adverse_effects'].replace({'Yes': 1, 'No': 0})
    return df

df = numerical_adverse_effects(df)
mean_adverse_effects = df['adverse_effects'].mean(numeric_only = True)
print(mean_adverse_effects)

0.09466418319272488


C:\Users\AsusIran\AppData\Local\Temp\ipykernel_31068\2005296984.py:3: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[:, 'adverse_effects'] = df['adverse_effects'].replace({'Yes': 1, 'No': 0})


In [31]:
df_drug = numerical_adverse_effects(df_drug)
df_placebo = numerical_adverse_effects(df_placebo)

C:\Users\AsusIran\AppData\Local\Temp\ipykernel_31068\2005296984.py:3: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[:, 'adverse_effects'] = df['adverse_effects'].replace({'Yes': 1, 'No': 0})


In [39]:
def diff_calc(df_drug, df_placebo, col):
    mean_wbc_drug = df_drug[col].mean(numeric_only = True)
    mean_wbc_placebo = df_placebo[col].mean(numeric_only = True)
    return (mean_wbc_drug - mean_wbc_placebo)


diff_mean_wbc = diff_calc(df_drug, df_placebo, 'wbc')
print("difference of mean in wbc between drug and placebo df: ", diff_mean_wbc)

diff_mean_rbc = diff_calc(df_drug, df_placebo, 'rbc')
print("difference of mean in rbc between drug and placebo df: ", diff_mean_rbc)

diff_mean_num_effects = diff_calc(df_drug, df_placebo, 'num_effects')
print("difference of mean in num effects between drug and placebo df: ", diff_mean_num_effects)

diff_mean_adverse_effects = diff_calc(df_drug, df_placebo, 'adverse_effects')
print("difference of mean in adverse_effects between drug and placebo df: ", diff_mean_adverse_effects)


    

difference of mean in wbc between drug and placebo df:  -0.028910170485221265
difference of mean in rbc between drug and placebo df:  0.01858756715194776
difference of mean in num effects between drug and placebo df:  0.0049178132772843275
difference of mean in adverse_effects between drug and placebo df:  0.003885420588570082


Perform Hypothesis Testing

As you can see in the output, for the p-value being 0.05, all the tests fail, meaning that the H0 is accepted and there is no significant difference in wbc, rbc, num_effects and adverse_effects between the drug and placebo groups.

In [41]:
from scipy.stats import ttest_ind

def perform_t_test(df_drug, df_placebo, col):
    if col == "wbc":
        alternative = 'less'
    else :
        alternative = 'greater'
    t_stat, p_value = ttest_ind(df_drug[col], df_placebo[col], equal_var = True, alternative = alternative)
    return t_stat, p_value

metrics = ['wbc', 'rbc', 'num_effects', 'adverse_effects']

for metric in metrics:
    t_stat, p_value = perform_t_test(df_drug, df_placebo, metric)
    print(f"T-test for {metric}: t-stat = {t_stat:.4f}, p-value = {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"  → Reject H₀: Significant difference in {metric} between Drug and Placebo groups.")
    else:
        print(f"  → Fail to reject H₀: No significant difference in {metric}.")

T-test for wbc: t-stat = -0.6559, p-value = 0.2559
  → Fail to reject H₀: No significant difference in wbc.
T-test for rbc: t-stat = 1.8367, p-value = 0.0331
  → Reject H₀: Significant difference in rbc between Drug and Placebo groups.
T-test for num_effects: t-stat = 0.6915, p-value = 0.2446
  → Fail to reject H₀: No significant difference in num_effects.
T-test for adverse_effects: t-stat = 0.6012, p-value = 0.2739
  → Fail to reject H₀: No significant difference in adverse_effects.


As for the p-value being set to 0.1, the tests for wbc, num_effects and adverse_effects will fail but there is a significant difference detected in rbc between two groups.

In [42]:
for metric in metrics:
    t_stat, p_value = perform_t_test(df_drug, df_placebo, metric)
    print(f"T-test for {metric}: t-stat = {t_stat:.4f}, p-value = {p_value:.4f}")
    
    if p_value < 0.1:
        print(f"  → Reject H₀: Significant difference in {metric} between Drug and Placebo groups.")
    else:
        print(f"  → Fail to reject H₀: No significant difference in {metric}.")

T-test for wbc: t-stat = -0.6559, p-value = 0.2559
  → Fail to reject H₀: No significant difference in wbc.
T-test for rbc: t-stat = 1.8367, p-value = 0.0331
  → Reject H₀: Significant difference in rbc between Drug and Placebo groups.
T-test for num_effects: t-stat = 0.6915, p-value = 0.2446
  → Fail to reject H₀: No significant difference in num_effects.
T-test for adverse_effects: t-stat = 0.6012, p-value = 0.2739
  → Fail to reject H₀: No significant difference in adverse_effects.


Interpretation: RBC count showed some level of significance when the p-value was set to 0.1. But when the p-value was 0.05 (stricter), the difference was not significant. This is showing that the difference between RBC count might not be very strong or conclusive.
As for WBC, num_effects and adverse_effects, there is no evidennce of a significant different between drug and placebo groups.

Some Additional info on p-value: is a probability that helps us determine whether the results of a hypothesis test are statistically significant or if they could have happened by chance. Thw common threshhold for this in most scientific studies is 0.05. Meaning that if the stat is below 0.05, we reject H0 and otherwise, fail to reject H0.

**equal-var argument**: 

If True, perform a standard independent 2 sample test that assumes equal population variances. If False, perform Welch’s t-test, which does not assume equal population variance. 
Because the standard deviation of Drug and Placebo group were near (calculated when I calculated the stats for each group), the equal-var was set to True.


**alternative argument**:

‘two-sided’: the means of the distributions underlying the samples are unequal.

‘less’: the mean of the distribution underlying the first sample is less than the mean of the distribution underlying the second sample.

‘greater’: the mean of the distribution underlying the first sample is greater than the mean of the distribution underlying the second sample.

Because in all of the 4 metrics, exept wbc, the *Drug mean* is more than the *Placebo mean*, the alternative argument for all of them exept wbc, should be greater. The alternative for wbc should be set to less.